# 03. MERFISH Gene Panel Inspection

This notebook characterizes the 300-gene MERFISH panel and identifies which 
ligand-receptor pairs from standard CCC databases are usable for analysis.

Key questions:
1. What genes are in the MERFISH panel?
2. How many L-R pairs from CellChatDB have both ligand and receptor in the panel?
3. Which developmental signaling pathways are well-represented vs. sparse?
4. What's the practical scope of CCC analysis given these constraints?

Outputs:
- `data/processed/merfish_gene_panel.csv`: the 300 genes
- `data/processed/usable_lr_pairs.csv`: L-R pairs with both genes detectable

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

from src.data_loading import get_project_root, load_main_merfish

PROJECT_ROOT = get_project_root()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/sydneycole/neuro/neuro


In [2]:
# We only need .var.index — load a small per-region file to save time
# (any of the MERFISH files has the same 300-gene panel)
adata = ad.read_h5ad(DATA_RAW / 'gw34_umb5900_ba17.h5ad', backed='r')

merfish_genes = list(adata.var.index)
print(f"MERFISH panel size: {len(merfish_genes)} genes")
print(f"\nFirst 30 genes (alphabetical):")
for g in sorted(merfish_genes)[:30]:
    print(f"  {g}")

# Save for later reference
pd.DataFrame({'gene': sorted(merfish_genes)}).to_csv(
    DATA_PROCESSED / 'merfish_gene_panel.csv', index=False
)
print(f"\nSaved gene panel to {DATA_PROCESSED / 'merfish_gene_panel.csv'}")

MERFISH panel size: 300 genes

First 30 genes (alphabetical):
  AC008124.1
  AC009506.1
  AC016065.1
  AC093249.6
  ACSBG1
  ACTN2
  ADAMTS5
  ADAMTSL3
  ADRA2A
  AGT
  AHI1
  AKAP7
  AKR1C2
  ALDOC
  ALPK1
  ANGPT2
  ANGPTL1
  ANKFN1
  ANOS1
  ANTXR2
  AP000866.1
  AP002748.4
  AQP4
  ARFGEF3
  ARHGAP5
  ARHGAP5-AS1
  ARID2
  ARRDC1-AS1
  ASH1L
  ASH1L-AS1

Saved gene panel to /Users/sydneycole/neuro/neuro/data/processed/merfish_gene_panel.csv


In [3]:
# Manually curated categories based on developmental neuroscience knowledge
# This isn't exhaustive — just to give a sense of what kinds of genes are in the panel

categories = {
    'Excitatory neuron markers': ['NEUROD2', 'NEUROD6', 'SATB2', 'BCL11B', 'CTIP2', 'FEZF2', 
                                   'TBR1', 'CUX1', 'CUX2', 'POU3F2', 'POU3F3', 'FOXP2'],
    'Inhibitory neuron markers': ['GAD1', 'GAD2', 'DLX1', 'DLX2', 'DLX5', 'DLX6', 'LHX6',
                                   'SST', 'PVALB', 'VIP', 'CALB2', 'NPY'],
    'Progenitor markers': ['SOX2', 'PAX6', 'NES', 'VIM', 'GLI3', 'EMX1', 'EMX2', 
                           'EOMES', 'TBR2', 'NEUROG2', 'ASCL1', 'HES1', 'HES5'],
    'Wnt signaling': ['WNT3', 'WNT3A', 'WNT5A', 'WNT7A', 'WNT7B', 'FZD1', 'FZD2', 'FZD3',
                      'FZD7', 'FZD9', 'LEF1', 'TCF7', 'TCF7L1', 'TCF7L2', 'CTNNB1'],
    'Notch signaling': ['DLL1', 'DLL3', 'JAG1', 'JAG2', 'NOTCH1', 'NOTCH2', 'NOTCH3',
                        'HES1', 'HES5', 'HEY1', 'HEY2'],
    'FGF signaling': ['FGF2', 'FGF8', 'FGFR1', 'FGFR2', 'FGFR3', 'SPRY1', 'SPRY2'],
    'BMP signaling': ['BMP2', 'BMP4', 'BMP7', 'BMPR1A', 'BMPR1B', 'BMPR2', 'ID1', 'ID2', 'ID3'],
    'Axon guidance': ['SLIT1', 'SLIT2', 'SLIT3', 'ROBO1', 'ROBO2', 'EFNB1', 'EFNB2',
                      'EPHB1', 'EPHB2', 'EPHA4', 'SEMA3A', 'SEMA6A', 'PLXNA2', 'NTNG1'],
    'Reelin pathway': ['RELN', 'VLDLR', 'LRP8', 'DAB1'],
    'Migration': ['CXCL12', 'CXCR4', 'CXCR7', 'NRG1', 'NRG3', 'ERBB4'],
    'Glia': ['GFAP', 'AQP4', 'ALDH1L1', 'SOX9', 'OLIG1', 'OLIG2', 'PDGFRA', 'NG2'],
    'Vascular': ['PECAM1', 'CLDN5', 'CDH5', 'VWF', 'PDGFRB', 'CSPG4'],
}

# Check which markers are actually present
print("Genes in MERFISH panel by category:\n")
present_by_category = {}
for cat, genes in categories.items():
    present = [g for g in genes if g in merfish_genes]
    missing = [g for g in genes if g not in merfish_genes]
    present_by_category[cat] = present
    print(f"{cat}:")
    print(f"  In panel ({len(present)}): {present}")
    print(f"  Missing ({len(missing)}): {missing}")
    print()

Genes in MERFISH panel by category:

Excitatory neuron markers:
  In panel (9): ['NEUROD2', 'NEUROD6', 'SATB2', 'BCL11B', 'FEZF2', 'TBR1', 'CUX1', 'CUX2', 'FOXP2']
  Missing (3): ['CTIP2', 'POU3F2', 'POU3F3']

Inhibitory neuron markers:
  In panel (6): ['GAD2', 'LHX6', 'SST', 'PVALB', 'VIP', 'NPY']
  Missing (6): ['GAD1', 'DLX1', 'DLX2', 'DLX5', 'DLX6', 'CALB2']

Progenitor markers:
  In panel (8): ['SOX2', 'PAX6', 'GLI3', 'EMX1', 'EOMES', 'NEUROG2', 'HES1', 'HES5']
  Missing (5): ['NES', 'VIM', 'EMX2', 'TBR2', 'ASCL1']

Wnt signaling:
  In panel (0): []
  Missing (15): ['WNT3', 'WNT3A', 'WNT5A', 'WNT7A', 'WNT7B', 'FZD1', 'FZD2', 'FZD3', 'FZD7', 'FZD9', 'LEF1', 'TCF7', 'TCF7L1', 'TCF7L2', 'CTNNB1']

Notch signaling:
  In panel (3): ['HES1', 'HES5', 'HEY1']
  Missing (8): ['DLL1', 'DLL3', 'JAG1', 'JAG2', 'NOTCH1', 'NOTCH2', 'NOTCH3', 'HEY2']

FGF signaling:
  In panel (0): []
  Missing (7): ['FGF2', 'FGF8', 'FGFR1', 'FGFR2', 'FGFR3', 'SPRY1', 'SPRY2']

BMP signaling:
  In panel (0): []


In [4]:
# Inside the gene panel inspection notebook
import commot as ct

# Load CellChat secreted signaling
df_lr_secreted = ct.pp.ligand_receptor_database(
    species='human', signaling_type='Secreted Signaling', database='CellChat'
)
print(f"Secreted L-R pairs: {len(df_lr_secreted)}")

# Load ECM-receptor (probably your richest category)
df_lr_ecm = ct.pp.ligand_receptor_database(
    species='human', signaling_type='ECM-Receptor', database='CellChat'
)
print(f"ECM-receptor pairs: {len(df_lr_ecm)}")

# Cell-cell contact
df_lr_cc = ct.pp.ligand_receptor_database(
    species='human', signaling_type='Cell-Cell Contact', database='CellChat'
)
print(f"Cell-cell contact pairs: {len(df_lr_cc)}")

Secreted L-R pairs: 1199
ECM-receptor pairs: 421
Cell-cell contact pairs: 319


In [6]:
def filter_lr_by_panel(df_lr, panel_genes):
    """
    Filter L-R pairs to those with both ligand and receptor in the panel.
    
    COMMOT's L-R format typically has columns ['0', '1', '2', '3'] where:
    - col 0: ligand (may be a complex like 'WNT3A_WNT5A' for heterodimers)
    - col 1: receptor (may be a complex like 'FZD1_LRP6')
    - col 2: pathway
    - col 3: annotation
    
    For complexes, we check whether ALL subunits are in the panel.
    """
    panel_set = set(panel_genes)
    
    def all_subunits_in_panel(gene_str):
        if pd.isna(gene_str):
            return False
        # Subunits separated by '_' for complexes
        subunits = str(gene_str).split('_')
        return all(s in panel_set for s in subunits)
    
    # Check ligand and receptor presence
    ligand_col = df_lr.columns[0]
    receptor_col = df_lr.columns[1]
    
    ligand_present = df_lr[ligand_col].apply(all_subunits_in_panel)
    receptor_present = df_lr[receptor_col].apply(all_subunits_in_panel)
    
    both_present = ligand_present & receptor_present
    
    return df_lr[both_present].copy(), {
        'total': len(df_lr),
        'ligand_only': (ligand_present & ~receptor_present).sum(),
        'receptor_only': (~ligand_present & receptor_present).sum(),
        'both': both_present.sum(),
        'neither': (~ligand_present & ~receptor_present).sum(),
    }

# Apply to each database
for name, df in [('Secreted', df_lr_secreted), 
                  ('Cell-Cell', df_lr_cc), 
                  ('ECM-Receptor', df_lr_ecm)]:
    usable, stats = filter_lr_by_panel(df, merfish_genes)
    print(f"\n{name} signaling:")
    for k, v in stats.items():
        print(f"  {k}: {v}")


Secreted signaling:
  total: 1199
  ligand_only: 25
  receptor_only: 8
  both: 0
  neither: 1166

Cell-Cell signaling:
  total: 319
  ligand_only: 6
  receptor_only: 2
  both: 1
  neither: 310

ECM-Receptor signaling:
  total: 421
  ligand_only: 34
  receptor_only: 0
  both: 0
  neither: 387


In [7]:
# Inspect actual entries in the ECM-Receptor database
print("First 20 ECM-Receptor pairs:")
print(df_lr_ecm.head(20).to_string())

print("\nColumn names:")
print(df_lr_ecm.columns.tolist())

# Look at entries involving ITGA2 specifically
print("\nEntries containing ITGA2:")
itga2_mask = df_lr_ecm.apply(lambda row: 'ITGA2' in str(row.values), axis=1)
print(df_lr_ecm[itga2_mask].head(10).to_string())

First 20 ECM-Receptor pairs:
           0            1         2             3
1199  COL1A1  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1200  COL1A2  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1201  COL2A1  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1202  COL4A1  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1203  COL4A2  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1204  COL4A3  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1205  COL4A4  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1206  COL4A5  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1207  COL4A6  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1208  COL6A1  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1209  COL6A2  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1210  COL6A3  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1211  COL6A5  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1212  COL6A6  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1213  COL9A1  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1214  COL9A2  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1215  COL9A3  ITGA1_ITGB1  COLLAGEN  ECM-Receptor
1216  COL1A1  ITGA2_ITGB1  COLLAGEN  ECM-Receptor
1217  COL1A2  ITGA2_I

In [8]:
# Find all single-gene receptors in each database (no _ in the receptor field)
for name, df in [('Secreted', df_lr_secreted), 
                 ('Cell-Cell', df_lr_cc), 
                 ('ECM-Receptor', df_lr_ecm)]:
    single_receptors = df[~df['1'].str.contains('_', na=False)]['1'].unique()
    in_panel = [r for r in single_receptors if r in merfish_genes]
    print(f"\n{name}: {len(in_panel)} single-gene receptors in panel:")
    for r in in_panel:
        # For each such receptor, find ligands that are also in the panel
        receptor_rows = df[df['1'] == r]
        for _, row in receptor_rows.iterrows():
            ligand = row['0']
            ligand_subunits = ligand.split('_')
            if all(s in merfish_genes for s in ligand_subunits):
                print(f"  USABLE: {ligand} -> {r} ({row['2']})")


Secreted: 5 single-gene receptors in panel:

Cell-Cell: 3 single-gene receptors in panel:
  USABLE: CNTN2 -> CNTN2 (CNTN)

ECM-Receptor: 0 single-gene receptors in panel:


In [9]:
import anndata as ad
norm_exp = ad.read_h5ad(DATA_RAW / 'norm_exp.h5ad', backed='r')

print(f"Shape: {norm_exp.shape}")
print(f"obs columns: {list(norm_exp.obs.columns)}")
print(f"obsm keys: {list(norm_exp.obsm.keys())}")
print(f"var columns: {list(norm_exp.var.columns)}")
print(f"layers: {list(norm_exp.layers.keys())}")
print(f"uns keys: {list(norm_exp.uns.keys())}")

# Check the expression matrix details
sample = norm_exp.X[:1000, :].toarray() if hasattr(norm_exp.X, 'toarray') else norm_exp.X[:1000, :]
print(f"\nExpression matrix stats (first 1000 cells):")
print(f"  Shape: {sample.shape}")
print(f"  Min: {sample.min():.3f}")
print(f"  Max: {sample.max():.3f}")
print(f"  Mean: {sample.mean():.3f}")
print(f"  Std: {sample.std():.3f}")
print(f"  Sparse: {hasattr(norm_exp.X, 'toarray')}")

Shape: (5700964, 300)
obs columns: ['gw', 'sample', 'region', 'H1_cluster', 'H2_cluster', 'H3_cluster', 'H1_annotation', 'H2_annotation', 'H3_annotation', 'area', 'layer']
obsm keys: ['spatial']
var columns: []
layers: []
uns keys: []

Expression matrix stats (first 1000 cells):
  Shape: (1000, 300)
  Min: 0.000
  Max: 0.562
  Mean: 0.008
  Std: 0.022
  Sparse: False
